In [42]:
from cProfile import label

import pandas as pd
import numpy as np
import country_converter as coco

In [4]:
df = pd.read_csv('hotel_bookings.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [5]:
columns_remove = ['adults', 'children', 'babies', 'meal', 'previous_bookings_not_canceled', 'assigned_room_type', 'deposit_type', 'required_car_parking_spaces', 'days_in_waiting_list','reservation_status_date', 'reservation_status']

cancellation = df.drop(columns_remove , axis=1)

cancellation.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 21 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   hotel                      119390 non-null  str    
 1   is_canceled                119390 non-null  int64  
 2   lead_time                  119390 non-null  int64  
 3   arrival_date_year          119390 non-null  int64  
 4   arrival_date_month         119390 non-null  str    
 5   arrival_date_week_number   119390 non-null  int64  
 6   arrival_date_day_of_month  119390 non-null  int64  
 7   stays_in_weekend_nights    119390 non-null  int64  
 8   stays_in_week_nights       119390 non-null  int64  
 9   country                    118902 non-null  str    
 10  market_segment             119390 non-null  str    
 11  distribution_channel       119390 non-null  str    
 12  is_repeated_guest          119390 non-null  int64  
 13  previous_cancellations     119390 non-nu

In [6]:
cancellation['arrival_date'] = pd.to_datetime(
    cancellation['arrival_date_year'].astype(str) + '-' +
    cancellation['arrival_date_month'] + '-' +
    cancellation['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

cancellation = cancellation.drop(['arrival_date_year','arrival_date_month','arrival_date_day_of_month'], axis=1)


In [7]:
cancellation.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   hotel                      119390 non-null  str           
 1   is_canceled                119390 non-null  int64         
 2   lead_time                  119390 non-null  int64         
 3   arrival_date_week_number   119390 non-null  int64         
 4   stays_in_weekend_nights    119390 non-null  int64         
 5   stays_in_week_nights       119390 non-null  int64         
 6   country                    118902 non-null  str           
 7   market_segment             119390 non-null  str           
 8   distribution_channel       119390 non-null  str           
 9   is_repeated_guest          119390 non-null  int64         
 10  previous_cancellations     119390 non-null  int64         
 11  reserved_room_type         119390 non-null  str           
 12 

In [8]:
cancellation['stays_in_week_nights'].unique()

array([ 0,  1,  2,  3,  4,  5, 10, 11,  8,  6,  7, 15,  9, 12, 33, 20, 14,
       16, 21, 13, 30, 19, 24, 40, 22, 42, 50, 25, 17, 32, 26, 18, 34, 35,
       41])

In [9]:
cancellation[cancellation.duplicated(keep=False)]

,hotel,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,reserved_room_type,booking_changes,agent,company,customer_type,adr,total_of_special_requests,arrival_date
4,Resort Hotel,0,14,27,0,2,GBR,Online TA,TA/TO,0,0,A,0,240.0,NaN,Transient,98.00,1,2015-07-01
5,Resort Hotel,0,14,27,0,2,GBR,Online TA,TA/TO,0,0,A,0,240.0,NaN,Transient,98.00,1,2015-07-01
21,Resort Hotel,0,72,27,2,4,PRT,Direct,Direct,0,0,A,1,250.0,NaN,Transient,84.67,1,2015-07-01
22,Resort Hotel,0,72,27,2,4,PRT,Direct,Direct,0,0,A,1,250.0,NaN,Transient,84.67,1,2015-07-01
39,Resort Hotel,0,70,27,2,3,ROU,Direct,Direct,0,0,E,0,250.0,NaN,Transient,137.00,1,2015-07-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119352,City Hotel,0,63,35,0,3,SWE,Online TA,TA/TO,0,0,D,0,9.0,NaN,Transient-Party,195.33,2,2017-08-31
119353,City Hotel,0,63,35,0,3,SWE,Online TA,TA/TO,0,0,D,0,9.0,NaN,Transient-Party,195.33,2,2017-08-31
119354,City Hotel,0,63,35,0,3,SWE,Online TA,TA/TO,0,0,D,0,9.0,NaN,Transient-Party,195.33,2,2017-08-31
119372,City Hotel,0,175,35,1,3,NLD,Offline TA/TO,TA/TO,0,0,A,0,42.0,NaN,Transient,82.35,1,2017-08-31


In [11]:
df_cancellation = cancellation.drop_duplicates()
df_cancellation.info()

<class 'pandas.DataFrame'>
Index: 85524 entries, 0 to 119389
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   hotel                      85524 non-null  str           
 1   is_canceled                85524 non-null  int64         
 2   lead_time                  85524 non-null  int64         
 3   arrival_date_week_number   85524 non-null  int64         
 4   stays_in_weekend_nights    85524 non-null  int64         
 5   stays_in_week_nights       85524 non-null  int64         
 6   country                    85079 non-null  str           
 7   market_segment             85524 non-null  str           
 8   distribution_channel       85524 non-null  str           
 9   is_repeated_guest          85524 non-null  int64         
 10  previous_cancellations     85524 non-null  int64         
 11  reserved_room_type         85524 non-null  str           
 12  booking_changes    

# Target variable understand

In [12]:
df_cancellation['is_canceled'].value_counts(normalize=True)

is_canceled
0    0.724311
1    0.275689
Name: proportion, dtype: float64

In [13]:
df_cancellation.groupby('is_canceled')['adr'].mean()

is_canceled
0    102.430132
1    118.069487
Name: adr, dtype: float64

# Revenue impact

In [14]:
# Create variable
df_cancellation['total_nights'] = df_cancellation['stays_in_weekend_nights'] + df_cancellation['stays_in_week_nights']

df_cancellation['revenue'] = df_cancellation['adr'] * df_cancellation['total_nights']

#Total revenue lost due to cancellations
df_cancellation.groupby('is_canceled')['revenue'].sum()

is_canceled
0    22656881.30
1    11323273.16
Name: revenue, dtype: float64

# Time-based analysis (When cancel)

In [62]:
df_cancellation['arrival_date'].max()

Timestamp('2017-08-31 00:00:00')

In [16]:
df_cancellation.groupby(
    df_cancellation['arrival_date'].dt.month
)['is_canceled'].mean()

arrival_date
1     0.220627
2     0.231913
3     0.244354
4     0.303770
5     0.291447
6     0.304638
7     0.318658
8     0.322554
9     0.247735
10    0.240902
11    0.210494
12    0.267963
Name: is_canceled, dtype: float64

In [17]:
df_cancellation.groupby(pd.cut(
    df_cancellation['lead_time'],
    bins = [0,7,30,90,365]
))['is_canceled'].mean()

lead_time
(0, 7]       0.097077
(7, 30]      0.254275
(30, 90]     0.320973
(90, 365]    0.369678
Name: is_canceled, dtype: float64

In [18]:
df_cancellation.groupby(df_cancellation['market_segment'])['is_canceled'].mean()

market_segment
Aviation         0.198238
Complementary    0.128280
Corporate        0.120571
Direct           0.147074
Groups           0.283333
Offline TA/TO    0.148603
Online TA        0.352406
Undefined        1.000000
Name: is_canceled, dtype: float64

# Segment-based analysis (Who cancel)

In [19]:
df_cancellation.groupby(df_cancellation['market_segment'])['is_canceled'].mean()

market_segment
Aviation         0.198238
Complementary    0.128280
Corporate        0.120571
Direct           0.147074
Groups           0.283333
Offline TA/TO    0.148603
Online TA        0.352406
Undefined        1.000000
Name: is_canceled, dtype: float64

In [21]:
df_cancellation.groupby(df_cancellation['distribution_channel'])['is_canceled'].mean()

distribution_channel
Corporate    0.128331
Direct       0.148183
GDS          0.191011
TA/TO        0.310475
Undefined    0.800000
Name: is_canceled, dtype: float64

In [22]:
df_cancellation.groupby(df_cancellation['customer_type'])['is_canceled'].mean()

customer_type
Contract           0.163305
Group              0.100372
Transient          0.300291
Transient-Party    0.152746
Name: is_canceled, dtype: float64

In [23]:
df_cancellation.groupby(df_cancellation['is_repeated_guest'])['is_canceled'].mean()

is_repeated_guest
0    0.283810
1    0.077083
Name: is_canceled, dtype: float64

In [24]:
df_cancellation.groupby(df_cancellation['previous_cancellations'])['is_canceled'].mean()

previous_cancellations
0     0.267845
1     0.757465
2     0.303571
3     0.262295
4     0.200000
5     0.105263
6     0.117647
11    0.074074
13    0.750000
14    1.000000
19    1.000000
21    1.000000
24    1.000000
25    1.000000
26    1.000000
Name: is_canceled, dtype: float64

In [25]:
df_cancellation.groupby(df_cancellation['total_of_special_requests'])['is_canceled'].mean()

total_of_special_requests
0    0.334555
1    0.224685
2    0.212010
3    0.171889
4    0.106918
5    0.055556
Name: is_canceled, dtype: float64

In [26]:
df_cancellation.groupby(df_cancellation['hotel'])['is_canceled'].mean()

hotel
City Hotel      0.301050
Resort Hotel    0.235811
Name: is_canceled, dtype: float64

In [27]:
df_cancellation.groupby(df_cancellation['country'])['is_canceled'].mean()

country
ABW    0.000000
AGO    0.563050
AIA    0.000000
ALB    0.181818
AND    0.714286
         ...   
VGB    1.000000
VNM    0.250000
ZAF    0.371795
ZMB    0.500000
ZWE    0.500000
Name: is_canceled, Length: 177, dtype: float64

In [37]:
df_cancellation.groupby(df_cancellation['country'])['is_canceled'].sum().sort_values(ascending=False)

country
PRT    9543
GBR    1958
ESP    1838
FRA    1712
ITA    1053
       ... 
SLE       0
SMR       0
SLV       0
SUR       0
UGA       0
Name: is_canceled, Length: 177, dtype: int64

In [38]:
df_cancellation[df_cancellation['country'] == 'AUS']['is_canceled'].sum()

np.int64(91)

In [39]:
df_cancellation[df_cancellation['country'] == 'AUS']['is_canceled'].mean()

np.float64(0.2413793103448276)

# Data for PowerBI

In [15]:
df_cancellation.info()

<class 'pandas.DataFrame'>
Index: 85524 entries, 0 to 119389
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   hotel                      85524 non-null  str           
 1   is_canceled                85524 non-null  int64         
 2   lead_time                  85524 non-null  int64         
 3   arrival_date_week_number   85524 non-null  int64         
 4   stays_in_weekend_nights    85524 non-null  int64         
 5   stays_in_week_nights       85524 non-null  int64         
 6   country                    85079 non-null  str           
 7   market_segment             85524 non-null  str           
 8   distribution_channel       85524 non-null  str           
 9   is_repeated_guest          85524 non-null  int64         
 10  previous_cancellations     85524 non-null  int64         
 11  reserved_room_type         85524 non-null  str           
 12  booking_changes    

In [30]:
df_cancellation['realized_revenue'] = df_cancellation.apply(
    lambda row: row['revenue'] if row['is_canceled'] == 0 else 0,
    axis =1
)

df_cancellation['realized_revenue']

0            0.00
1            0.00
2           75.00
3           75.00
4          196.00
           ...   
119385     672.98
119386    1578.01
119387    1103.97
119388     730.80
119389    1360.80
Name: realized_revenue, Length: 85524, dtype: float64

In [32]:
df_cancellation['revenue_lost'] = df_cancellation.apply(
    lambda row: row['revenue'] if row['is_canceled'] == 1 else 0,
    axis = 1
)
df_cancellation[df_cancellation['revenue_lost'] != 0 ]

,hotel,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,country,market_segment,distribution_channel,is_repeated_guest,...,agent,company,customer_type,adr,total_of_special_requests,arrival_date,total_nights,revenue,realized_revenue,revenue_lost
8,Resort Hotel,1,85,27,0,3,PRT,Online TA,TA/TO,0,...,240.0,NaN,Transient,82.0,1,2015-07-01,3,246.0,0.0,246.0
9,Resort Hotel,1,75,27,0,3,PRT,Offline TA/TO,TA/TO,0,...,15.0,NaN,Transient,105.5,0,2015-07-01,3,316.5,0.0,316.5
10,Resort Hotel,1,23,27,0,4,PRT,Online TA,TA/TO,0,...,240.0,NaN,Transient,123.0,0,2015-07-01,4,492.0,0.0,492.0
27,Resort Hotel,1,60,27,2,5,PRT,Online TA,TA/TO,0,...,240.0,NaN,Transient,107.0,2,2015-07-01,7,749.0,0.0,749.0
32,Resort Hotel,1,96,27,2,8,PRT,Direct,Direct,0,...,NaN,NaN,Transient,108.3,2,2015-07-01,10,1083.0,0.0,1083.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108829,City Hotel,1,25,18,2,1,FRA,Corporate,Corporate,1,...,NaN,450.0,Transient,125.0,0,2017-05-06,3,375.0,0.0,375.0
111355,City Hotel,1,4,23,1,0,PRT,Corporate,Corporate,1,...,NaN,238.0,Transient,65.0,0,2017-06-05,1,65.0,0.0,65.0
111924,City Hotel,1,7,22,0,1,PRT,Corporate,Corporate,1,...,NaN,238.0,Transient,65.0,0,2017-05-31,1,65.0,0.0,65.0
111925,City Hotel,1,6,29,1,0,PRT,Corporate,Corporate,1,...,NaN,238.0,Transient,65.0,0,2017-07-17,1,65.0,0.0,65.0


In [33]:
df_cancellation['cancellation_status'] = df_cancellation['is_canceled'].map(
    {0: 'Completed',
     1: 'Canceled',}
)

In [57]:
df_cancellation['lead_time_buckets'] = pd.cut(
    df_cancellation['lead_time'],
    bins = [0,7,30,90,365, df_cancellation['lead_time'].max()],
    labels= ['0-7 days', '8-30 days', '31-90 days', '90+days', '1+year'],
    include_lowest=True
)

df_cancellation['lead_time_buckets'].value_counts(dropna=False)

lead_time_buckets
90+days       28633
31-90 days    22276
0-7 days      18009
8-30 days     16081
1+year          525
Name: count, dtype: int64

In [68]:
df_cancellation['total_nights_buckets'] = pd.cut(
    df_cancellation['total_nights'],
    bins = [0,3,7,14,28,60, df_cancellation['total_nights'].max()],
    labels= ['0-3 days', '4-7 days', '1+week', '2+weeks', '3+weeks', '2+months'],
    include_lowest=True
)
df_cancellation['total_nights_buckets'].value_counts(dropna=False)

total_nights_buckets
0-3 days    50036
4-7 days    30656
1+week       4463
2+weeks       321
3+weeks        47
2+months        1
Name: count, dtype: int64

In [66]:
df_cancellation['arrival_month'] = df_cancellation['arrival_date'].dt.to_period('M')
df_cancellation['arrival_month'].value_counts().sort_index()

arrival_month
2015-07    1634
2015-08    2391
2015-09    2659
2015-10    2538
2015-11    1596
2015-12    1939
2016-01    1813
2016-02    2743
2016-03    3778
2016-04    3691
2016-05    3670
2016-06    3431
2016-07    3809
2016-08    4372
2016-09    3743
2016-10    4112
2016-11    3283
2016-12    3099
2017-01    2783
2017-02    3242
2017-03    3617
2017-04    4055
2017-05    4503
2017-06    4201
2017-07    4483
2017-08    4339
Freq: M, Name: count, dtype: int64

In [36]:
df_cancellation['arrival_is_weekend'] = (
    df_cancellation['arrival_date'].dt.weekday >=5
).astype(int)

In [41]:
df_cancellation['is_domestic'] = np.where(
    df_cancellation['country'] == 'AUS',
    'Domestic',
    'International'
)

In [48]:
df_cancellation['country'] = df_cancellation['country'].fillna('Unknown')

df_cancellation['region'] = coco.convert(
    names = df_cancellation['country'],
    to = 'UNregion',
    not_found='Other'
)

Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
Unknown not found in regex
U

In [49]:
df_cancellation['region'].value_counts(dropna=False)

region
Southern Europe              37144
Western Europe               20594
Northern Europe              15568
South America                 2417
Eastern Europe                2295
Eastern Asia                  2248
Northern America              1852
Western Asia                   891
Other                          448
Australia and New Zealand      440
Northern Africa                384
Middle Africa                  362
Southern Asia                  260
South-eastern Asia             164
Central America                106
Eastern Africa                  97
Western Africa                  92
Southern Africa                 80
Caribbean                       48
Central Asia                    25
Micronesia                       3
Melanesia                        2
Polynesia                        2
Antarctica                       2
Name: count, dtype: int64

In [69]:
df_cancellation.info()

<class 'pandas.DataFrame'>
Index: 85524 entries, 0 to 119389
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   hotel                      85524 non-null  str           
 1   is_canceled                85524 non-null  int64         
 2   lead_time                  85524 non-null  int64         
 3   arrival_date_week_number   85524 non-null  int64         
 4   stays_in_weekend_nights    85524 non-null  int64         
 5   stays_in_week_nights       85524 non-null  int64         
 6   country                    85524 non-null  str           
 7   market_segment             85524 non-null  str           
 8   distribution_channel       85524 non-null  str           
 9   is_repeated_guest          85524 non-null  int64         
 10  previous_cancellations     85524 non-null  int64         
 11  reserved_room_type         85524 non-null  str           
 12  booking_changes    

In [71]:
df_cancellation.to_csv('hotel_demand_dataset.csv', index=False)

In [72]:
draft = pd.read_csv("hotel_demand_dataset.csv")

draft.info()

<class 'pandas.DataFrame'>
RangeIndex: 85524 entries, 0 to 85523
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   hotel                      85524 non-null  str    
 1   is_canceled                85524 non-null  int64  
 2   lead_time                  85524 non-null  int64  
 3   arrival_date_week_number   85524 non-null  int64  
 4   stays_in_weekend_nights    85524 non-null  int64  
 5   stays_in_week_nights       85524 non-null  int64  
 6   country                    85524 non-null  str    
 7   market_segment             85524 non-null  str    
 8   distribution_channel       85524 non-null  str    
 9   is_repeated_guest          85524 non-null  int64  
 10  previous_cancellations     85524 non-null  int64  
 11  reserved_room_type         85524 non-null  str    
 12  booking_changes            85524 non-null  int64  
 13  agent                      73759 non-null  float64
 14  c